# Phase 2B Exploratory Data-Quality Audit

## tl;dr

The closed-file v4-only audit contains **1 complete market**, **53,807 BTC ticks**, **211,229 valid PM states**, and **407 overlapping ≥1bp shock anchors**. Evidence maturity is **TINY_SAMPLE**; B2 is **INSUFFICIENT_STD0_EVENTS**. These outputs do not support causal, alpha, profitability, execution, or trading claims.

## Context & Methods

This notebook is a read-only companion to the versioned JSON/Parquet artifacts. It checks grain, collector version, deterministic ordering, valid-book selection, raw lineage population, and bounded result tables.

### Key Assumptions

- Only formal full-lifecycle `phase2a_prospective_v4` markets are eligible.
- Exchange timestamp is the primary research clock; receive time remains available for clock sensitivity.
- Shock anchors overlap and are descriptive, not independent statistical observations.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
report_path = Path(r'''D:\工作\量化\std0-quant\data\reports\phase2b_research_20260824T181517Z.json''')
report = json.loads(report_path.read_text(encoding='utf-8'))
timeline_path = Path(r'''D:\工作\量化\std0-quant\data\derived\phase2b\market_timeline_20260824T181517Z.parquet''')
grid_path = Path(r'''D:\工作\量化\std0-quant\data\derived\phase2b\market_grids_20260824T181517Z.parquet''')
corr_path = Path(r'''D:\工作\量化\std0-quant\data\derived\phase2b\cross_correlation_20260824T181517Z.parquet''')
response_path = Path(r'''D:\工作\量化\std0-quant\data\derived\phase2b\event_response_20260824T181517Z.parquet''')

## Data

### 1. Load bounded columns and confirm grain

In [2]:
timeline = pd.read_parquet(timeline_path, columns=['condition_id','source','event_timestamp_ms','receive_timestamp_ms','collector_version','book_valid','raw_file','raw_line'])
grids = pd.read_parquet(grid_path)
quality = pd.DataFrame({
    'metric':['timeline_rows','markets','btc_rows','pm_rows','v4_share','missing_raw_reference','grid_rows','grid_versions'],
    'value':[len(timeline),timeline.condition_id.nunique(),(timeline.source=='BTC').sum(),(timeline.source=='PM').sum(),(timeline.collector_version=='phase2a_prospective_v4').mean(),timeline[['raw_file','raw_line']].isna().any(axis=1).sum(),len(grids),sorted(grids.grid_ms.unique().tolist())]
})
display(quality)

,metric,value
0,timeline_rows,265036
1,markets,1
2,btc_rows,53807
3,pm_rows,211229
4,v4_share,1.0
5,missing_raw_reference,0
6,grid_rows,5100
7,grid_versions,"[100, 250, 500, 1000]"


### 2. Ordering and valid-state checks

In [3]:
checks = {
 'timeline_sorted': timeline[['event_timestamp_ms','receive_timestamp_ms','source']].reset_index(drop=True).equals(timeline.sort_values(['event_timestamp_ms','receive_timestamp_ms','source'])[['event_timestamp_ms','receive_timestamp_ms','source']].reset_index(drop=True)),
 'v4_only': bool((timeline.collector_version=='phase2a_prospective_v4').all()),
 'pm_rows_valid': bool(timeline.loc[timeline.source=='PM','book_valid'].fillna(False).all()),
 'raw_lineage_complete': int(timeline[['raw_file','raw_line']].isna().any(axis=1).sum()) == 0,
 'phase2a_frozen': report['phase2a_frozen_invariants']['status'] == 'PASS',
 'sha_failures': len(report['recorder_cohort_state']['raw_integrity']['sha256_failures'])
}
checks

C:\Users\Administrator\AppData\Local\Temp\ipykernel_6436\50210580.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  'pm_rows_valid': bool(timeline.loc[timeline.source=='PM','book_valid'].fillna(False).all()),


{'timeline_sorted': True,
 'v4_only': True,
 'pm_rows_valid': True,
 'raw_lineage_complete': True,
 'phase2a_frozen': True,
 'sha_failures': 0}

## Results

### 3. Lead-lag summaries

In [4]:
corr = pd.read_parquet(corr_path).sort_values('lag_ms')
response = pd.read_parquet(response_path).sort_values(['shock_bucket','horizon_ms'])
display(corr.nlargest(7, 'correlation')[['lag_ms','n','correlation']])
display(response.head(18))

,lag_ms,n,correlation
9,250,702,0.381600
8,0,703,0.268881
10,500,701,0.106156
2,-1500,704,0.068428
3,-1250,704,0.064102
7,-250,704,0.062088
0,-2000,704,0.041932


,condition_id,shock_bucket,horizon_ms,n,signed_mean_response,signed_median_response
0,0x035600d8e35f2f451d167d51e94e860cc9dbf02a64ea...,1-2bp,100,143,0.001049,0.00
1,0x035600d8e35f2f451d167d51e94e860cc9dbf02a64ea...,1-2bp,250,143,0.000909,0.00
2,0x035600d8e35f2f451d167d51e94e860cc9dbf02a64ea...,1-2bp,500,143,0.000944,0.00
3,0x035600d8e35f2f451d167d51e94e860cc9dbf02a64ea...,1-2bp,1000,143,0.002343,0.00
4,0x035600d8e35f2f451d167d51e94e860cc9dbf02a64ea...,1-2bp,2000,140,0.002036,0.00
5,0x035600d8e35f2f451d167d51e94e860cc9dbf02a64ea...,1-2bp,5000,133,0.005602,0.00
6,0x035600d8e35f2f451d167d51e94e860cc9dbf02a64ea...,2-5bp,100,74,0.001689,0.00
7,0x035600d8e35f2f451d167d51e94e860cc9dbf02a64ea...,2-5bp,250,73,0.001849,0.00
8,0x035600d8e35f2f451d167d51e94e860cc9dbf02a64ea...,2-5bp,500,72,0.000833,0.00
9,0x035600d8e35f2f451d167d51e94e860cc9dbf02a64ea...,2-5bp,1000,71,-0.002113,0.00


### 4. Coverage and missingness by grid

In [5]:
grid_quality = grids.groupby('grid_ms').agg(rows=('timestamp_ms','size'), book_valid_rate=('book_valid','mean'), btc_missing_rate=('btc_price',lambda s:s.isna().mean()), pm_missing_rate=('pm_mid',lambda s:s.isna().mean())).reset_index()
display(grid_quality)

,grid_ms,rows,book_valid_rate,btc_missing_rate,pm_missing_rate
0,100,3000,0.589667,0.000333,0.410333
1,250,1200,0.590000,0.000833,0.410000
2,500,600,0.590000,0.001667,0.410000
3,1000,300,0.590000,0.003333,0.410000


## Takeaways

- The artifact is v4-only, ordered, raw-referenced, and built from closed SHA-verified files.
- Row-level invalid PM states are excluded before grids and response estimation; the report retains their exclusion count.
- A one-market peak correlation lag is a mechanism hint only. It is not general evidence and must be revisited as markets accumulate.
- B2 remains disabled until the primary cohort naturally contains a fully-covered lineage/PIT-passing std0 observation.
- Phase 2A thresholds and hashes remain unchanged.